# Q2 Classification — Ionosphere（手刻 DNN：Backpropagation + SGD）

- 只用 `numpy`（運算）、`pandas`（讀 CSV）、`matplotlib`（畫圖）；前向／反向傳播、Softmax、Cross-entropy、SGD + momentum 全部手寫。
- 80% 訓練 / 20% 測試（分層抽樣）；訓練集內再切 15% 為驗證集，**只用於 early stopping**，測試集不參與任何選擇。
- 本版重點：**梯度檢查**、**CV 與最終訓練流程一致（都用 early stopping）**、**每個 latent 維度各自選超參數**、**混淆矩陣**、**補充圖含 early-stopping 實際採用的 epoch**。

## 0. 匯入套件與全域設定

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

plt.rcParams["axes.unicode_minus"] = False

DATA_PATH = "2026_ionosphere_data.csv"   # 與本 notebook 放同一資料夾

LATENT_DIMS = [2, 5, 10]                 # (c) 比較的 latent 節點數
SNAPSHOTS_AT = [10, 100, 500, 1000]      # (c) 主圖的訓練階段
SNAPSHOTS_SUP = [5, 10, 25, 50]          # (c) 補充圖的訓練階段（另外一定會加上 early stopping 採用的 epoch）
MAIN_SEED = 42
N_SEEDS = 10                             # 多次切分次數（除錯可改 3）
CV_FOLDS = 3
BATCH = 32
USE_ENSEMBLE = True                      # 額外討論用；不需要可關掉省時間
ENSEMBLE_K = 5

# 基準模型：SGD、無正則化、ReLU latent
BASE = dict(hidden=[32, 16], drop=0.0, l2=0.0, lr=0.05, mom=0.0,
            act="relu", cosine=False, epochs=300)
VIS_CFG = {**BASE, "epochs": 1000}       # (c) 主圖：刻意不正則化，觀察 latent 演變

# 改良版：momentum + cosine lr + tanh latent + dropout + L2 + early stopping
COMMON = dict(lr=0.02, mom=0.9, act="tanh", cosine=True, epochs=200, patience=40)
GRID = [{**COMMON, **g} for g in [
    dict(hidden=[32, 16], drop=0.0, l2=1e-2),
    dict(hidden=[32, 16], drop=0.2, l2=1e-2),
    dict(hidden=[64, 32], drop=0.3, l2=1e-2),
    dict(hidden=[64, 32], drop=0.3, l2=3e-2),
]]


def cfg_str(c):
    return f"hidden={c['hidden']}, dropout={c['drop']}, L2={c['l2']}"

## 1. 資料讀取與切分

In [ ]:
df = pd.read_csv(DATA_PATH, header=None)
X_all = df.iloc[:, :-1].to_numpy(dtype=np.float64)
y_all = (df.iloc[:, -1].to_numpy() == "g").astype(int)   # g=1, b=0
N_FEAT = X_all.shape[1]
print("資料大小:", df.shape, "| 特徵數:", N_FEAT, "| g:", (y_all == 1).sum(), "| b:", (y_all == 0).sum())


def stratified_split(y, frac, rng):
    """每個類別各自抽 frac 比例，回傳 (剩下的 idx, 抽出的 idx)"""
    rest, picked = [], []
    for c in np.unique(y):
        idx = rng.permutation(np.where(y == c)[0])
        k = int(round(len(idx) * frac))
        picked.append(idx[:k])
        rest.append(idx[k:])
    return rng.permutation(np.concatenate(rest)), rng.permutation(np.concatenate(picked))


def make_data(seed):
    """分層 80/20；訓練集再切 15% 當驗證集；標準化只用 train 統計量"""
    rng = np.random.RandomState(seed)
    rest, te = stratified_split(y_all, 0.2, rng)
    tr_pos, va_pos = stratified_split(y_all[rest], 0.15, rng)
    tr, va = rest[tr_pos], rest[va_pos]
    mu, sd = X_all[tr].mean(0), X_all[tr].std(0)
    sd[sd == 0] = 1.0
    f = lambda i: (X_all[i] - mu) / sd
    return dict(Xtr=f(tr), ytr=y_all[tr], Xva=f(va), yva=y_all[va],
                Xte=f(te), yte=y_all[te])

## 2. (a) 手寫 DNN，最小化 Cross-entropy

網路：輸入 → 隱藏層(ReLU, dropout) → **latent 層**(tanh 或 ReLU) → 輸出層(Softmax)

$$E(\mathbf{w}) = -\sum_{n=1}^{N}\sum_{k=1}^{K} t_{nk}\log y_k(\mathbf{x}_n,\mathbf{w})$$

Softmax + Cross-entropy 時輸出層梯度化簡為 $\partial E/\partial z_k = y_k - t_k$（程式：`dZ = (a[-1]-Y)/m`），再以鏈鎖律逐層往前（ReLU 導數 $\mathbb{1}[z>0]$、tanh 導數 $1-a^2$），並加入 L2 與 momentum。
本版每個模型使用**自己的 `RandomState`**，不同模型之間的亂數互不影響。

In [ ]:
class MLP:
    """隱藏層 ReLU(+dropout) -> latent 層(tanh 或 ReLU) -> Softmax 輸出"""

    def __init__(self, sizes, cfg, rng):
        self.cfg, self.rng = cfg, rng
        self.L = len(sizes) - 1
        self.lr, self.l2, self.mom = cfg["lr"], cfg["l2"], cfg["mom"]
        self.p, self.act = cfg["drop"], cfg["act"]
        self.W, self.b = [], []
        for i in range(self.L):
            is_tanh_latent = (i == self.L - 2 and self.act == "tanh")
            scale = np.sqrt((1.0 if is_tanh_latent else 2.0) / sizes[i])   # He / Xavier
            self.W.append(rng.randn(sizes[i], sizes[i + 1]) * scale)
            self.b.append(np.zeros((1, sizes[i + 1])))
        self.vW = [np.zeros_like(w) for w in self.W]
        self.vb = [np.zeros_like(b) for b in self.b]

    @staticmethod
    def softmax(z):
        z = z - z.max(1, keepdims=True)          # 數值穩定
        e = np.exp(z)
        return e / e.sum(1, keepdims=True)

    def forward(self, X, train=False):
        a = X
        self.z, self.a, self.mask = [], [X], []
        for i in range(self.L):
            z = a @ self.W[i] + self.b[i]
            m = None
            if i == self.L - 1:
                a = self.softmax(z)
            elif i == self.L - 2:                # latent 層：不做 dropout
                a = np.tanh(z) if self.act == "tanh" else np.maximum(0, z)
            else:
                a = np.maximum(0, z)
                if train and self.p > 0:         # inverted dropout
                    m = (self.rng.rand(*a.shape) > self.p) / (1.0 - self.p)
                    a = a * m
            self.z.append(z)
            self.a.append(a)
            self.mask.append(m)
        return a

    def backward(self, Y):
        m = Y.shape[0]
        dZ = (self.a[-1] - Y) / m                # softmax + cross-entropy
        gW, gb = [None] * self.L, [None] * self.L
        for i in reversed(range(self.L)):
            gW[i] = self.a[i].T @ dZ + self.l2 * self.W[i]
            gb[i] = dZ.sum(0, keepdims=True)
            if i > 0:
                dA = dZ @ self.W[i].T
                k = i - 1
                if self.mask[k] is not None:
                    dA = dA * self.mask[k]
                if k == self.L - 2 and self.act == "tanh":
                    dZ = dA * (1.0 - self.a[k + 1] ** 2)
                else:
                    dZ = dA * (self.z[k] > 0)
        self.gW, self.gb = gW, gb                # 存起來供梯度檢查使用
        for i in range(self.L):                  # SGD + momentum
            self.vW[i] = self.mom * self.vW[i] - self.lr * gW[i]
            self.vb[i] = self.mom * self.vb[i] - self.lr * gb[i]
            self.W[i] += self.vW[i]
            self.b[i] += self.vb[i]

    def latent(self, X):
        self.forward(X)
        return self.a[-2]


def ce(p, y, eps=1e-9):
    return -np.mean(np.log(p[np.arange(len(y)), y] + eps))


def err(p, y):
    return np.mean(p.argmax(1) != y)


def _snapshot(model, d):
    Xtr, ytr, Xte, yte = d["Xtr"], d["ytr"], d["Xte"], d["yte"]
    return (model.latent(Xtr), model.latent(Xte),
            model.W[-1].copy(), model.b[-1].copy(),
            err(model.forward(Xtr), ytr), err(model.forward(Xte), yte))


def train(d, dim, seed, cfg, snapshots=(), track=True):
    """mini-batch SGD。回傳 model, 歷史紀錄, latent 快照, 最終結果。
    cfg['patience']>0 且資料含驗證集 -> 以驗證 loss 做 early stopping 並還原最佳權重，
    此時快照另存一份 snaps['best']（= 實際採用的權重）。"""
    rng = np.random.RandomState(seed)
    model = MLP([N_FEAT] + cfg["hidden"] + [dim, 2], cfg, rng)
    Xtr, ytr, Xte, yte = d["Xtr"], d["ytr"], d["Xte"], d["yte"]
    Ytr = np.eye(2)[ytr]
    n, E = len(ytr), cfg["epochs"]
    H = dict(loss=[], te_loss=[], va_loss=[], tr=[], te=[])
    snaps = {}
    patience = cfg.get("patience", 0)
    use_es = patience > 0 and "Xva" in d
    best_loss, best_ep, wait, best_state = np.inf, E, 0, None

    for ep in range(1, E + 1):
        if cfg["cosine"]:
            model.lr = cfg["lr"] * (0.01 + 0.99 * 0.5 * (1 + np.cos(np.pi * (ep - 1) / E)))
        perm = rng.permutation(n)
        for s in range(0, n, BATCH):
            idx = perm[s:s + BATCH]
            model.forward(Xtr[idx], train=True)
            model.backward(Ytr[idx])
        if track:
            ptr, pte = model.forward(Xtr), model.forward(Xte)
            H["loss"].append(ce(ptr, ytr)); H["te_loss"].append(ce(pte, yte))
            H["tr"].append(err(ptr, ytr));  H["te"].append(err(pte, yte))
        if ep in snapshots:
            snaps[ep] = _snapshot(model, d)
        if use_es:
            lva = ce(model.forward(d["Xva"]), d["yva"])
            H["va_loss"].append(lva)
            if lva < best_loss - 1e-4:
                best_loss, best_ep, wait = lva, ep, 0
                best_state = ([w.copy() for w in model.W], [b.copy() for b in model.b])
            else:
                wait += 1
                if wait >= patience:
                    break

    if use_es and best_state is not None:
        model.W, model.b = best_state            # 還原驗證 loss 最低的權重
    if len(snapshots) > 0:
        snaps["best"] = _snapshot(model, d)      # 最終採用權重的 latent 快照
    res = dict(train_err=err(model.forward(Xtr), ytr),
               test_err=err(model.forward(Xte), yte),
               stop_ep=best_ep if use_es else E)
    return model, H, snaps, res


RETRY_LOG = []   # 記錄 (c) 主圖是否發生 dead ReLU 換 seed（報告需如實說明）


def train_vis(d, dim, seed, cfg, snapshots, max_tries=5):
    """(c) 專用：ReLU latent 偶爾會 dead ReLU（訓練誤差很高）；發生時換 seed 重訓並記錄"""
    for t in range(max_tries):
        out = train(d, dim, seed + t, cfg, snapshots=snapshots, track=False)
        if out[3]["train_err"] < 0.2:
            break
        RETRY_LOG.append((dim, seed + t, out[3]["train_err"]))
        print(f"  [dim={dim}] seed={seed + t} 訓練誤差 {out[3]['train_err']:.3f} 偏高(dead ReLU)，換 seed 重訓")
    return out


def cv_select(d, seed, dim):
    """只用訓練資料做 stratified k-fold CV 挑超參數（不碰驗證集與測試集）。
    每個 fold 的訓練部分再切 15% 當內部驗證集，讓 CV 也使用 early stopping，
    與最終訓練流程一致；並針對指定的 latent 維度 dim 挑選。"""
    X, y = d["Xtr"], d["ytr"]
    rng = np.random.RandomState(seed + 1000)
    folds = [[] for _ in range(CV_FOLDS)]
    for c in np.unique(y):
        idx = rng.permutation(np.where(y == c)[0])
        for k, part in enumerate(np.array_split(idx, CV_FOLDS)):
            folds[k].extend(part.tolist())
    folds = [np.array(f) for f in folds]

    scores = []
    for gi, cfg in enumerate(GRID):
        errs, ces = [], []
        for k in range(CV_FOLDS):
            va = folds[k]
            tr_all = np.concatenate([folds[j] for j in range(CV_FOLDS) if j != k])
            r2 = np.random.RandomState(seed * 7 + k)
            tr_pos, iv_pos = stratified_split(y[tr_all], 0.15, r2)
            tr, iv = tr_all[tr_pos], tr_all[iv_pos]
            dk = dict(Xtr=X[tr], ytr=y[tr], Xva=X[iv], yva=y[iv], Xte=X[va], yte=y[va])
            m, _, _, r = train(dk, dim, seed * 10 + k, cfg, track=False)
            errs.append(r["test_err"])
            ces.append(ce(m.forward(X[va]), y[va]))
        scores.append((np.mean(errs), np.mean(ces)))
    best = min(range(len(GRID)), key=lambda i: (round(scores[i][0], 4), scores[i][1]))
    return best, scores


def ensemble_test_err(d, dim, seed, cfg, K=ENSEMBLE_K):
    P = 0
    for k in range(K):
        m, *_ = train(d, dim, seed * 100 + k, cfg, track=False)
        P = P + m.forward(d["Xte"])
    return err(P / K, d["yte"])

### 2.1 梯度檢查（證明手寫的 backpropagation 正確）

以數值微分 $\frac{E(w+\epsilon)-E(w-\epsilon)}{2\epsilon}$ 與反向傳播算出的梯度比較，相對誤差應遠小於 $10^{-5}$。（關閉 dropout 與 L2，測試 tanh 與 ReLU 兩種 latent。）

In [ ]:
def grad_check(act, eps=1e-5, n_pick=6):
    cfg_t = dict(hidden=[8, 6], drop=0.0, l2=0.0, lr=0.0, mom=0.0,
                 act=act, cosine=False, epochs=1)
    rng = np.random.RandomState(0)
    m = MLP([N_FEAT, 8, 6, 3, 2], cfg_t, rng)
    X = rng.randn(10, N_FEAT); y = rng.randint(0, 2, 10); Y = np.eye(2)[y]
    m.forward(X); m.backward(Y)                  # lr=0：權重不會被改動
    worst = 0.0
    for i in range(m.L):
        for _ in range(n_pick):
            a, b = rng.randint(m.W[i].shape[0]), rng.randint(m.W[i].shape[1])
            old = m.W[i][a, b]
            m.W[i][a, b] = old + eps; lp = ce(m.forward(X), y)
            m.W[i][a, b] = old - eps; lm = ce(m.forward(X), y)
            m.W[i][a, b] = old
            num, ana = (lp - lm) / (2 * eps), m.gW[i][a, b]
            worst = max(worst, abs(num - ana) / (abs(num) + abs(ana) + 1e-12))
    return worst


for act in ["tanh", "relu"]:
    w = grad_check(act)
    print(f"latent={act:4s}: 最大相對誤差 = {w:.2e}  ->  {'通過' if w < 1e-5 else '未通過，請檢查 backward'}")

### 2.2 (a) 訓練最終模型

對每個 latent 維度，各自用訓練資料做 CV（含 early stopping）選超參數；(b) 的主模型為 latent=2。

In [ ]:
d = make_data(MAIN_SEED)
print(f"Train={len(d['ytr'])}, Val={len(d['yva'])}, Test={len(d['yte'])}")

best_cfg = {}
for dim in LATENT_DIMS:
    bi, cv_scores = cv_select(d, MAIN_SEED, dim)
    best_cfg[dim] = GRID[bi]
    print(f"\n[CV 超參數選擇] latent_dim={dim}（只用訓練資料）")
    for i, (e_, l_) in enumerate(cv_scores):
        mark = "  <- 選中" if i == bi else ""
        print(f"  {cfg_str(GRID[i])}: CV err={e_:.4f}, CV loss={l_:.4f}{mark}")

cfg = best_cfg[2]                                  # (b) 主模型：latent=2
model, H, _, res = train(d, 2, MAIN_SEED, cfg)
print("\n主模型（latent=2）訓練完成。")

## 3. (b) 網路架構、學習曲線、訓練/測試錯誤率

In [ ]:
print("===== (1) Network architecture =====")
print(f"{N_FEAT} - {' - '.join(map(str, cfg['hidden']))} - 2(latent) - 2(output)")
print(f"  hidden layers: {len(cfg['hidden'])} 層，神經元數 {cfg['hidden']}，ReLU")
print(f"  latent layer : 2 個神經元，{cfg['act']}")
print(f"  output layer : 2 個神經元，Softmax")
print(f"  lr={cfg['lr']} (cosine decay), momentum={cfg['mom']}, dropout={cfg['drop']}, "
      f"L2={cfg['l2']}, batch={BATCH}")
print(f"  max epochs={cfg['epochs']}, early stopping patience={cfg['patience']}, "
      f"實際採用 epoch={res['stop_ep']}")

print("\n===== (3)(4) Error rate =====")
print(f"Train samples={len(d['ytr'])}, Test samples={len(d['yte'])}")
print(f"Training error rate = {res['train_err']:.4f}")
print(f"Test error rate     = {res['test_err']:.4f}")
print(f"(測試集每錯 1 筆約 {100/len(d['yte']):.2f}%)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(H["loss"], label="Train loss")
ax[0].plot(H["va_loss"], label="Validation loss")
ax[0].plot(H["te_loss"], label="Test loss (reference only)", alpha=0.6)
ax[0].axvline(res["stop_ep"] - 1, color="gray", ls=":", label=f"best epoch={res['stop_ep']}")
ax[0].set_title("(2) Learning Curve (Cross-Entropy Loss)")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].legend()

ax[1].plot(H["tr"], label="Train error")
ax[1].plot(H["te"], label="Test error")
ax[1].axvline(res["stop_ep"] - 1, color="gray", ls=":")
ax[1].set_title("Error Rate"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout()
plt.savefig("learning_curve_classification.png", dpi=150)
plt.show()

**說明**：選 epoch 只看 **validation loss**（灰色虛線為驗證 loss 最低處）；test 曲線只用來觀察是否過擬合，不參與任何選擇。

### 混淆矩陣與各類別錯誤率（測試集）

In [ ]:
def confusion(model, X, y, title):
    p = model.forward(X).argmax(1)
    tp = int(((p == 1) & (y == 1)).sum()); tn = int(((p == 0) & (y == 0)).sum())
    fp = int(((p == 1) & (y == 0)).sum()); fn = int(((p == 0) & (y == 1)).sum())
    print(f"[{title}] Confusion matrix（列=真實, 欄=預測, 順序 b,g）")
    print(f"        pred b  pred g\n  true b  {tn:5d}  {fp:5d}\n  true g  {fn:5d}  {tp:5d}")
    print(f"  b 類錯誤率 = {fp / max(tn + fp, 1):.3f}（{fp}/{tn + fp}）, "
          f"g 類錯誤率 = {fn / max(tp + fn, 1):.3f}（{fn}/{tp + fn}）")


confusion(model, d["Xtr"], d["ytr"], "Train")
confusion(model, d["Xte"], d["yte"], "Test")

## 4. (c) 不同 latent 節點數、不同訓練階段的 latent 分布

比較輸出層前一層節點數 = 2 / 5 / 10。維度 > 2 時以 **PCA（用訓練集）降到 2D**。

- **主圖**：無正則化、ReLU latent、1000 epochs，用來觀察 latent 分布的演變（與題目附圖設定相近）；加上**測試點（紅框三角形）**檢查泛化。
- **補充圖**：最終（正則化 + early stopping）模型，附決策邊界，最後一欄為 early stopping **實際採用的 epoch**，與 (b) 的模型一致。

In [ ]:
def plot_latent_simple(ax, Ztr, ytr, title, Zte=None, yte=None):
    """訓練點(圓) + 測試點(三角)；高維用『訓練集』的 PCA 投影"""
    if Ztr.shape[1] > 2:
        mu = Ztr.mean(0)
        _, _, Vt = np.linalg.svd(Ztr - mu, full_matrices=False)
        Ptr = (Ztr - mu) @ Vt[:2].T
        Pte = None if Zte is None else (Zte - mu) @ Vt[:2].T
        ax.set_xlabel("PC1 of latent features", fontsize=8)
        ax.set_ylabel("PC2 of latent features", fontsize=8)
    else:
        Ptr, Pte = Ztr, Zte
    for cls, col, name in [(0, "blue", "b"), (1, "lime", "g")]:
        ax.scatter(Ptr[ytr == cls, 0], Ptr[ytr == cls, 1], c=col, s=20,
                   edgecolors="k", linewidths=0.8, label=f"train {name}")
        if Pte is not None:
            ax.scatter(Pte[yte == cls, 0], Pte[yte == cls, 1], c=col, s=55,
                       marker="^", edgecolors="r", linewidths=1.0, label=f"test {name}")
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=6)


def plot_latent(ax, snap, ytr, yte, dim, title, legend=False):
    """含測試點與決策邊界；dim>2 時 x 軸為決策方向 logit(g)-logit(b)"""
    Ztr, Zte, Wl, bl, _, _ = snap
    w = Wl[:, 1] - Wl[:, 0]
    c = bl[0, 1] - bl[0, 0]
    if dim == 2:
        Ptr, Pte = Ztr, Zte
        xl, yl = "latent unit 1", "latent unit 2"
    else:
        wn = w / (np.linalg.norm(w) + 1e-12)
        Rtr = Ztr - np.outer(Ztr @ wn, wn)
        Rte = Zte - np.outer(Zte @ wn, wn)
        mu = Rtr.mean(0, keepdims=True)
        _, _, Vt = np.linalg.svd(Rtr - mu, full_matrices=False)
        Ptr = np.c_[Ztr @ w + c, (Rtr - mu) @ Vt[0]]
        Pte = np.c_[Zte @ w + c, (Rte - mu) @ Vt[0]]
        xl, yl = "logit(g) - logit(b)", "PC1 (orthogonal to decision axis)"

    for cls, col, name in [(0, "navy", "b"), (1, "green", "g")]:
        ax.scatter(Ptr[ytr == cls, 0], Ptr[ytr == cls, 1], c=col, s=12, alpha=0.45,
                   label=f"{name} train")
        ax.scatter(Pte[yte == cls, 0], Pte[yte == cls, 1], c=col, s=40, marker="^",
                   edgecolors="k", linewidths=0.6, label=f"{name} test")
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    if dim == 2:
        if abs(w[1]) > 1e-9:
            xs = np.array([x0, x1])
            ax.plot(xs, -(w[0] * xs + c) / w[1], "r--", lw=1.3, label="decision boundary")
        elif abs(w[0]) > 1e-9:
            ax.axvline(-c / w[0], color="r", ls="--", lw=1.3, label="decision boundary")
    else:
        ax.axvline(0, color="r", ls="--", lw=1.3, label="decision boundary")
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel(xl, fontsize=8); ax.set_ylabel(yl, fontsize=8)
    if legend:
        ax.legend(fontsize=7, loc="best")

### (c) 主圖：無正則化、ReLU latent、1000 epochs（訓練點 + 測試點）

In [ ]:
RETRY_LOG.clear()
fig, axes = plt.subplots(len(LATENT_DIMS), len(SNAPSHOTS_AT),
                         figsize=(4.6 * len(SNAPSHOTS_AT), 4.0 * len(LATENT_DIMS)))
for r, dim in enumerate(LATENT_DIMS):
    _, _, snaps, rs = train_vis(d, dim, MAIN_SEED, VIS_CFG, SNAPSHOTS_AT)
    print(f"latent_dim={dim}: final train_err={rs['train_err']:.4f}, test_err={rs['test_err']:.4f}")
    for c_, ep in enumerate(SNAPSHOTS_AT):
        plot_latent_simple(axes[r, c_], snaps[ep][0], d["ytr"],
                           f"{dim}D feature {ep} epoch\n(train err={snaps[ep][4]:.3f}, test err={snaps[ep][5]:.3f})",
                           snaps[ep][1], d["yte"])
plt.tight_layout()
plt.savefig("compare_latent_dims.png", dpi=150)
plt.show()
print("\n[dead ReLU 重訓紀錄]", RETRY_LOG if RETRY_LOG else "無（三個維度皆使用 MAIN_SEED 直接訓練成功）")

**說明**：無正則化訓練 1000 epochs，訓練點（圓）後期分得很開，是網路逐漸記住訓練集的結果；泛化能力要看測試點（紅框三角形）是否落在對應類別區域，以及標題中的 train / test err。若訓練點分很開、測試點卻與另一類混雜，就是過擬合。

### (c) 補充圖：最終（正則化 + early stopping）模型，含決策邊界

In [ ]:
n_cols = len(SNAPSHOTS_SUP) + 1                       # 固定 epoch 欄 + 最後一欄「採用的 epoch」
fig, axes = plt.subplots(len(LATENT_DIMS), n_cols,
                         figsize=(4.6 * n_cols, 4.0 * len(LATENT_DIMS)))
for r, dim in enumerate(LATENT_DIMS):
    cfg_d = best_cfg[dim]
    _, _, snaps, rs = train(d, dim, MAIN_SEED, cfg_d, snapshots=SNAPSHOTS_SUP, track=False)
    cols = [ep for ep in SNAPSHOTS_SUP if ep in snaps and ep < rs["stop_ep"]] + ["best"]
    for c_ in range(n_cols):
        ax = axes[r, c_]
        if c_ >= len(cols):
            ax.axis("off"); continue
        key = cols[c_]
        s = snaps[key]
        name = f"epoch={rs['stop_ep']} (early-stop chosen)" if key == "best" else f"epoch={key}"
        plot_latent(ax, s, d["ytr"], d["yte"], dim,
                    f"dim={dim}, {name}\ntrain err={s[4]:.3f}, test err={s[5]:.3f}",
                    legend=(r == 0 and c_ == 0))
plt.tight_layout()
plt.savefig("compare_latent_dims_boundary.png", dpi=150)
plt.show()

## 5. 結果討論：多次切分對照

資料只有 351 筆，單一切分的 test error 波動很大。此處用 N_SEEDS 次不同切分；**每次切分、每個 latent 維度各自做 CV 選超參數**（無測試集洩漏），比較：

- **baseline**：SGD、無正則化、ReLU latent
- **improved**：momentum + cosine + tanh + dropout + L2 + early stopping
- **ensemble**：improved 的 K 個模型取平均（額外討論）

paired diff = baseline − improved，為正代表改良版較好；若 |diff| < 2×SE，改善不顯著。

In [ ]:
print(f"===== 多次切分對照（{N_SEEDS} 次）=====")
res_b = {k: [] for k in LATENT_DIMS}
res_i = {k: [] for k in LATENT_DIMS}
res_t = {k: [] for k in LATENT_DIMS}
res_e = {k: [] for k in LATENT_DIMS}
picked = {k: Counter() for k in LATENT_DIMS}

for s in range(N_SEEDS):
    ds = make_data(s)
    for dim in LATENT_DIMS:
        bi, _ = cv_select(ds, s, dim)
        picked[dim][bi] += 1
        res_b[dim].append(train(ds, dim, s, BASE, track=False)[3]["test_err"])
        r_ = train(ds, dim, s, GRID[bi], track=False)[3]
        res_i[dim].append(r_["test_err"])
        res_t[dim].append(r_["train_err"])
        if USE_ENSEMBLE:
            res_e[dim].append(ensemble_test_err(ds, dim, s, GRID[bi]))
    print(f"  seed {s} 完成")

print("\nCV 選中的超參數次數（各 latent 維度）：")
for dim in LATENT_DIMS:
    print(f"  dim={dim}: " + "; ".join(f"[{cfg_str(GRID[i])}] x{n_}" for i, n_ in sorted(picked[dim].items())))

print(f"\n{'dim':>4} | {'baseline':>15} | {'improved':>15} | "
      f"{'paired diff (base-imp)':>24} | {'ensemble':>15} | train err")
for dim in LATENT_DIMS:
    b, i_, t = map(np.array, (res_b[dim], res_i[dim], res_t[dim]))
    diff = b - i_
    se = diff.std(ddof=1) / np.sqrt(len(diff))
    ens = (f"{np.mean(res_e[dim]):.4f} ± {np.std(res_e[dim]):.4f}" if USE_ENSEMBLE else "-")
    print(f"{dim:>4} | {b.mean():.4f} ± {b.std():.4f} | {i_.mean():.4f} ± {i_.std():.4f} | "
          f"{diff.mean():+.4f} ± {se:.4f} (SE)   | {ens} | {t.mean():.4f}")
print("\n（paired diff 為正代表改良版較好；若 |diff| 小於 2 倍 SE，改善不顯著）")

### 結論（請依實際輸出填寫）

1. **(a)** 梯度檢查最大相對誤差見 2.1，確認手寫 backpropagation 正確。
2. **(b)** 架構、train / test error、混淆矩陣見第 3 節。
3. **(c)** latent 節點數 2 / 5 / 10 的分布隨 epoch 的變化，以及測試點是否與訓練點同區，見第 4 節。
4. **過擬合**：以 train / test 差距、validation loss 曲線與多次切分的平均 ± 標準差判斷。
5. 若第 4 節出現 dead ReLU 換 seed，須在報告中說明。